# 02 Data Cleaning

**Objective:** Transform the raw Hospital Readmissions dataset into a clean, consistent, and analysis-ready dataset. Every transformation must be justified with reasoning to ensure data integrity and traceability.

## 1. Data Loading

**Why:** We must first import the necessary libraries and load the raw data. Using `pandas` allows us to manipulate structured data efficiently.

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_csv('../data/raw/hospital_readmissions_30k.csv')
print(f"Raw dataset shape: {df.shape}")

Raw dataset shape: (30600, 12)


## 2. Initial Inspection

**Why:** Before cleaning, we need to understand the structure of the data, the column names, and the existing data types to identify areas that need attention.

In [3]:
display(df.head())
display(df.info())

,patient_id,age,gender,blood_pressure,cholesterol,bmi,diabetes,hypertension,medication_count,length_of_stay,discharge_destination,readmitted_30_days
0,1.0,74.0,other,130/72,240.0,31.5,yes,no,5.0,1.0,nursing_facility,yes
1,2.0,46.0,Female,120/92,292.0,36.3,No,No,4.0,3.0,Nursing_Facility,No
2,3.0,89.0,other,135/78,153.0,30.3,no,yes,1.0,1.0,home,no
3,4.0,84.0,Female,123/80,NaN,31.5,No,Yes,3.0,10.0,Home,No
4,5.0,32.0,Other,135/84,205.0,18.4,No,Yes,6.0,4.0,Nursing_Facility,No


<class 'pandas.DataFrame'>
RangeIndex: 30600 entries, 0 to 30599
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   patient_id             29081 non-null  float64
 1   age                    29070 non-null  float64
 2   gender                 29285 non-null  str    
 3   blood_pressure         29271 non-null  str    
 4   cholesterol            29104 non-null  float64
 5   bmi                    29030 non-null  float64
 6   diabetes               29373 non-null  str    
 7   hypertension           29279 non-null  str    
 8   medication_count       29107 non-null  float64
 9   length_of_stay         29132 non-null  float64
 10  discharge_destination  29332 non-null  str    
 11  readmitted_30_days     29232 non-null  str    
dtypes: float64(6), str(6)
memory usage: 2.8 MB


None

## 3. Column Standardization

**Why:** Standardizing column names to a uniform format (lowercase, no spaces, snake_case) guarantees programmatic consistency and avoids bugs caused by minor typos.

In [4]:
# Strip whitespace, lower case, replace spaces with underscores
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print("Standardized columns:", df.columns.tolist())

Standardized columns: ['patient_id', 'age', 'gender', 'blood_pressure', 'cholesterol', 'bmi', 'diabetes', 'hypertension', 'medication_count', 'length_of_stay', 'discharge_destination', 'readmitted_30_days']


## 4. Duplicate Handling

**Why:** Duplicate records distort statistical analysis and model training by inflating the importance of repeated observations. We check for and remove any fully identical rows.

In [5]:
initial_rows = len(df)
duplicates_count = df.duplicated().sum()
print(f"Found {duplicates_count} duplicate rows.")

if duplicates_count > 0:
    df = df.drop_duplicates()
    print(f"Removed {duplicates_count} duplicates. Current shape: {df.shape}")

Found 376 duplicate rows.
Removed 376 duplicates. Current shape: (30224, 12)


## 5. Inconsistent Value Handling

**Why:** Values entered across different systems or by humans often suffer from casing issues and formatting inconsistencies. By coercing categorical columns to lower-case and fixing blood pressure formatting, we ensure value integrity.

In [6]:
categorical_cols = df.select_dtypes(include=['object']).columns

# Standardize strings to lower case and strip whitespace
for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()
    # Standardize 'nan' strings back to actual NaNs
    df[col] = df[col].replace('nan', np.nan)

# specific handling for blood_pressure which might be formatted as 'systolic/diastolic'
if 'blood_pressure' in df.columns:
    # Creating two new columns directly by splitting the variable
    bp_split = df['blood_pressure'].astype(str).str.split('/', expand=True)
    df['systolic_bp'] = pd.to_numeric(bp_split[0], errors='coerce')
    df['diastolic_bp'] = pd.to_numeric(bp_split[1], errors='coerce')
    # Drop the original non-numeric column
    df = df.drop('blood_pressure', axis=1)
    print("Split 'blood_pressure' into 'systolic_bp' and 'diastolic_bp'.")

Split 'blood_pressure' into 'systolic_bp' and 'diastolic_bp'.


## 6. Data Type Fixing

**Why:** Pandas sometimes infers incorrect datatypes (e.g., float instead of int, or object instead of numeric) due to nulls or dirty strings. We must explicitly convert numeric concepts into numeric formats for aggregation and modeling.

In [7]:
numeric_features_to_fix = ['age', 'cholesterol', 'bmi', 'medication_count', 'length_of_stay']
for col in numeric_features_to_fix:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print("Data types after fixing:")
print(df.dtypes)

Data types after fixing:
patient_id               float64
age                      float64
gender                       str
cholesterol              float64
bmi                      float64
diabetes                     str
hypertension                 str
medication_count         float64
length_of_stay           float64
discharge_destination        str
readmitted_30_days           str
systolic_bp              float64
diastolic_bp             float64
dtype: object


## 7. Outlier Detection and Handling

**Why:** Clinical data can have severe outliers (e.g., age > 100, extreme BMI) that might represent data entry errors. We apply clinical thresholds to detect these and cap them (Winsorization) or remove them if entirely unrealistic, preserving distribution while limiting extreme influence.

In [8]:
print("--- Detecting Unrealistic Values ---")
# Handle Age
if 'age' in df.columns:
    unrealistic_age = len(df[(df['age'] < 0) | (df['age'] > 110)])
    print(f"Found {unrealistic_age} patients with age < 0 or > 110. Converting to NaN for imputation.")
    df.loc[(df['age'] < 0) | (df['age'] > 110), 'age'] = np.nan

# Handle BMI
if 'bmi' in df.columns:
    unrealistic_bmi = len(df[(df['bmi'] < 10) | (df['bmi'] > 80)])
    print(f"Found {unrealistic_bmi} patients with unrealistic BMI (<10 or >80). Converting to NaN.")
    df.loc[(df['bmi'] < 10) | (df['bmi'] > 80), 'bmi'] = np.nan

# Handling statistical outliers with IQR clipping instead of removal to limit data loss
def clip_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    return series.clip(lower=lower_bound, upper=upper_bound)

numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns
for col in numerical_cols:
    if col not in ['patient_id']: # Do not clip IDs
        df[col] = clip_outliers(df[col])
print("Applied IQR capping to continuous numerical columns.")

--- Detecting Unrealistic Values ---
Found 304 patients with age < 0 or > 110. Converting to NaN for imputation.
Found 304 patients with unrealistic BMI (<10 or >80). Converting to NaN.
Applied IQR capping to continuous numerical columns.


## 8. Missing Value Treatment

**Why:** Many machine learning models fail when exposed to missing values. We use statistical imputation to retain data:
- **Numerical Columns**: Imputed with the MEDIAN because it is robust against outliers.
- **Categorical Columns**: Imputed with the MODE (most frequent category) to represent the most likely state.

In [9]:
print("Missing values before treatment:")
print(df.isnull().sum())

# Treatment Strategy
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns
categorical_cols_all = df.select_dtypes(include=['object', 'category']).columns

for col in numerical_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

for col in categorical_cols_all:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print("\nMissing values after treatment:")
print(df.isnull().sum())

Missing values before treatment:
patient_id               1499
age                      1815
gender                   1493
cholesterol              1474
bmi                      1848
diabetes                 1422
hypertension             1515
medication_count         1477
length_of_stay           1446
discharge_destination    1472
readmitted_30_days       1560
systolic_bp              1514
diastolic_bp             1514
dtype: int64

Missing values after treatment:
patient_id               0
age                      0
gender                   0
cholesterol              0
bmi                      0
diabetes                 0
hypertension             0
medication_count         0
length_of_stay           0
discharge_destination    0
readmitted_30_days       0
systolic_bp              0
diastolic_bp             0
dtype: int64


## 9. Feature Engineering

**Why:** Creating derived features gives machine learning models easier access to relationships within the data, adding predictive power.
- `is_readmitted`: Clean boolean binary flag representing our target variable.
- `age_group`: Bucketing age into clinical phases.
- `stay_category`: Categorizing duration bounds.
- `risk_tier`: Composite feature to surface high impact cohorts directly.

In [10]:
# is_readmitted (binary flag)
if 'readmitted_30_days' in df.columns:
    df['is_readmitted'] = df['readmitted_30_days'].apply(lambda x: 1 if str(x).lower() == 'yes' else 0)

# age_group
def map_age(age):
    if age < 18: return 'child'
    elif age < 35: return 'young'
    elif age < 65: return 'adult'
    else: return 'senior'
df['age_group'] = df['age'].apply(map_age)

# stay_category
def map_stay(los):
    if los <= 3: return 'short'
    elif los <= 7: return 'medium'
    elif los <= 14: return 'long'
    else: return 'critical'
df['stay_category'] = df['length_of_stay'].apply(map_stay)

# risk_tier (Low, Medium, High risk based on readmission history/age)
# Simulated logical tiering for readmission risk
def map_risk(row):
    if row['age'] > 65 and row['length_of_stay'] > 7:
        return 'high'
    elif row['age'] > 50 or row['length_of_stay'] > 4:
        return 'medium'
    else:
        return 'low'
df['risk_tier'] = df.apply(map_risk, axis=1)

display(df[['is_readmitted', 'age_group', 'stay_category', 'risk_tier']].head())

,is_readmitted,age_group,stay_category,risk_tier
0,1,senior,short,medium
1,0,adult,short,low
2,0,senior,short,medium
3,0,senior,long,high
4,0,young,medium,low


## 10. Final Validation

**Why:** To ensure we haven't introduced any issues, we must perform a final cross-check on data shapes, missing value prevalence, and distributions.

In [11]:
print(f"Final Dataset Shape: {df.shape}")

if df.isnull().sum().sum() == 0:
    print("Success: No missing values remain in the dataset.")
else:
    print("Warning: Some missing values remain!")
    
display(df.describe(include='all'))

Final Dataset Shape: (30224, 17)
Success: No missing values remain in the dataset.


,patient_id,age,gender,cholesterol,bmi,diabetes,hypertension,medication_count,length_of_stay,discharge_destination,readmitted_30_days,systolic_bp,diastolic_bp,is_readmitted,age_group,stay_category,risk_tier
count,30224.000000,30224.000000,30224,30224.000000,30224.00000,30224,30224,30224.000000,30224.000000,30224,30224,30224.000000,30224.000000,30224.000000,30224,30224,30224
unique,NaN,NaN,3,NaN,NaN,2,2,NaN,NaN,3,2,NaN,NaN,NaN,3,4,3
top,NaN,NaN,male,NaN,NaN,no,no,NaN,NaN,home,no,NaN,NaN,NaN,adult,medium,medium
freq,NaN,NaN,11178,NaN,NaN,15873,15895,NaN,NaN,21477,26711,NaN,NaN,NaN,13463,12842,22489
mean,16365.300490,53.861501,NaN,226.753375,28.95753,NaN,NaN,5.134595,5.630393,NaN,NaN,134.978395,85.036627,0.116232,NaN,NaN,NaN
std,15867.806259,20.424781,NaN,44.912193,6.14924,NaN,NaN,3.297502,2.959465,NaN,NaN,14.303306,8.688464,0.320508,NaN,NaN,NaN
min,1.000000,18.000000,NaN,150.000000,18.00000,NaN,NaN,0.000000,1.000000,NaN,NaN,110.000000,70.000000,0.000000,NaN,NaN,NaN
25%,7998.750000,37.000000,NaN,190.000000,23.80000,NaN,NaN,2.000000,3.000000,NaN,NaN,123.000000,78.000000,0.000000,NaN,NaN,NaN
50%,15148.000000,54.000000,NaN,226.000000,28.90000,NaN,NaN,5.000000,6.000000,NaN,NaN,135.000000,85.000000,0.000000,NaN,NaN,NaN
75%,22321.250000,71.000000,NaN,262.000000,34.20000,NaN,NaN,8.000000,8.000000,NaN,NaN,147.000000,92.000000,0.000000,NaN,NaN,NaN


## 11. Dataset Export

**Why:** The final clean environment state needs to be serialized for the downstream machine learning pipeline or analysts to pick up from.

In [12]:
import os
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/cleaned_data.csv', index=False)
print("Cleaned dataset successfully saved to '../data/processed/cleaned_data.csv'.")

Cleaned dataset successfully saved to '../data/processed/cleaned_data.csv'.


## 12. Cleaning Summary
>
**Transformations Applied:**
- Converted column names to lower_case and removed whitespace.
- Stripped and lowercased categorical string values to fix formatting misalignments (e.g. Yes/yes/YES).
- Separated compound `blood_pressure` strings into `systolic_bp` and `diastolic_bp` continuous variables.
- Removed exact duplicate rows from the original data slice.
- Detected and transformed unclinical numeric values (like age > 110) into NAs prior to imputation.
- Managed minor statistical outliers by clipping continuous variables to their upper/lower IQR bounds.
- Conducted global imputation: medians for continuous logic and modes for discretes.

**Engineered Features:**
- `is_readmitted`: Logical binary target variable.
- `age_group`: Groups patients temporally (child, young, adult, senior).
- `stay_category`: Groups patients procedurally (short, medium, long, critical).
- `risk_tier`: Multi-variate risk dimension (high, medium, low) defined by age and stay boundaries.

**Key Decisions:**
- Did not drop rows with missing values to maximize dataset richness; leveraged robust imputations to bridge gaps.
- Outliers handled via clipping instead of removal to reflect accurate healthcare variance distributions without throwing off non-robust models.

In [13]:
import pandas as pd

raw = pd.read_csv('../data/raw/hospital_readmissions_30k.csv')
cleaned = pd.read_csv('../data/processed/cleaned_data.csv')

print("Raw rows:", len(raw))
print("Cleaned rows:", len(cleaned))
print("Duplicated rows in cleaned:", cleaned.duplicated().sum())

# Check if it matches raw after dropping dupes
print("Cleaned after drop_duplicates:", cleaned.drop_duplicates().shape)



Raw rows: 30600
Cleaned rows: 30224
Duplicated rows in cleaned: 214
Cleaned after drop_duplicates: (30010, 17)
